# 🔬 Fitting All Distributions in Climatology Engine

This notebook demonstrates how to fit all available distributions.

**What you will learn:**
- Loading distribution plugins
- Fitting all distributions on sample data
- Comparing results using AICc
- Selecting the best statistical model
- Plotting AICc comparison chart

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins

sns.set_style('whitegrid')
print('✅ Libraries loaded.')

In [ ]:
# Load distribution plugins
plugins = load_plugins()
print(f'✅ Number of loaded distributions: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

# Store in dictionary for easy access
distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# Load sample data
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# Select tmean data for one year (365 days)
data_year = data[:365, 1]

print(f'📊 Number of samples: {len(data_year)}')
print(f'   Mean: {np.mean(data_year):.2f}°C')
print(f'   Standard deviation: {np.std(data_year):.2f}°C')
print(f'   Minimum: {np.min(data_year):.2f}°C')
print(f'   Maximum: {np.max(data_year):.2f}°C')

In [ ]:
# Fit all distributions
results_all = {}
print("\n🔄 Fitting distributions...\n")

for name, dist in distributions.items():
    try:
        res = dist.fit(data_year)
        results_all[name] = res
        print(f"✅ {name}: AICc = {res.get('aicc', np.nan):.2f}, "
              f"BIC = {res.get('bic', np.nan):.2f}")
    except Exception as e:
        print(f"❌ {name}: Error - {str(e)}")
        results_all[name] = None

print("\n✅ All distributions fitted successfully.")

In [ ]:
# Select the best model (minimum AICc)
valid_results = {k: v for k, v in results_all.items() 
                  if v is not None and 'aicc' in v and not np.isnan(v['aicc'])}

if valid_results:
    best_name = min(valid_results, key=lambda x: valid_results[x]['aicc'])
    best_result = valid_results[best_name]

    print("=" * 60)
    print(f"🏆 Best distribution: {best_name}")
    print(f"   AICc: {best_result['aicc']:.4f}")
    print(f"   BIC: {best_result.get('bic', np.nan):.4f}")
    print(f"   Log-likelihood: {best_result.get('loglik', np.nan):.4f}")
    print("=" * 60)
else:
    print("❌ No valid distribution found.")

In [ ]:
# Create comparison table
if valid_results:
    comparison_df = pd.DataFrame([{
        'Distribution': name,
        'AICc': res['aicc'],
        'BIC': res.get('bic', np.nan),
        'LogLik': res.get('loglik', np.nan),
        'ΔAICc': res['aicc'] - best_result['aicc'],
        'N_Params': res.get('n_params', np.nan)
    } for name, res in valid_results.items()])

    comparison_df = comparison_df.sort_values('AICc').reset_index(drop=True)
    comparison_df.index = comparison_df.index + 1
    comparison_df
else:
    print("❌ No data to display.")

In [ ]:
# Plot AICc comparison chart
if valid_results:
    fig, ax = plt.subplots(figsize=(10, 6))

    colors = ['#2ecc71' if name == best_name else '#e74c3c' 
              for name in comparison_df['Distribution']]
    bars = ax.barh(comparison_df['Distribution'], comparison_df['AICc'], 
                   color=colors, alpha=0.7, edgecolor='black', linewidth=1)

    ax.axvline(best_result['aicc'], color='black', linestyle='--', 
               linewidth=2, alpha=0.7, label=f'Best: {best_name}')
    
    ax.set_xlabel('AICc', fontsize=12)
    ax.set_title('AICc Comparison of Distributions', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')

    for i, (name, aicc) in enumerate(zip(comparison_df['Distribution'], comparison_df['AICc'])):
        ax.text(aicc + 0.5, i, f'{aicc:.1f}', va='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
else:
    print("❌ No data to plot.")

In [ ]:
# Display parameters of the best distribution
if valid_results:
    print(f"📊 Fitted parameters for {best_name} distribution:")
    print("=" * 50)
    for key, value in best_result.items():
        if isinstance(value, float):
            print(f"   {key}: {value:.6f}")
        else:
            print(f"   {key}: {value}")
    print("=" * 50)
else:
    print("❌ No data to display.")

In [ ]:
# Plot PDF curves of all models
if valid_results:
    fig, ax = plt.subplots(figsize=(12, 7))

    # Data histogram
    ax.hist(data_year, bins=30, density=True, alpha=0.3, 
            color='gray', edgecolor='black', label='Data')

    # Different colors for distributions
    colors_list = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    x = np.linspace(min(data_year), max(data_year), 500)

    for i, (name, res) in enumerate(valid_results.items()):
        try:
            # Calculate PDF for each distribution
            dist = distributions[name]
            if hasattr(dist, 'pdf'):
                params = {p: res[p] for p in dist.params if p in res}
                pdf_vals = dist.pdf(x, params)
                ax.plot(x, pdf_vals, color=colors_list[i % len(colors_list)], 
                        linewidth=2, label=f'{name} (AICc={res["aicc"]:.1f})')
            else:
                print(f"⚠️ Distribution {name} has no pdf method.")
        except Exception as e:
            print(f"⚠️ Error plotting {name}: {str(e)}")

    ax.set_xlabel('Temperature (°C)', fontsize=12)
    ax.set_ylabel('Probability Density', fontsize=12)
    ax.set_title('Fitted Distribution Curves Comparison', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("❌ No data to plot.")

## 📋 Summary

In this notebook you learned:

✅ Loading distribution plugins with `load_plugins()`
✅ Fitting all available distributions on sample data
✅ Comparing results using AICc criterion
✅ Selecting the best statistical model
✅ Plotting AICc comparison chart
✅ Plotting fitted distribution curves

---

**Next Steps:**
- Notebook 04: Model Selection (AIC, BIC, AICc)
- Notebook 05: Quality Control (Quality Flag)
- Notebook 06: Bootstrap Uncertainty

---

**Important Note:**
AICc is suitable for comparing models with different numbers of parameters. The lower the AICc, the better the model.